In [1]:
import sys
print(sys.executable)

c:\Users\shchitt\Downloads\Projects\domain-adapt\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import sqlite3

db = Path("../data/financial.sqlite").resolve()
assert db.is_file(), db
conn = sqlite3.connect(f"file:{db.as_posix()}?mode=ro", uri=True)

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
print("Tables:", [name for (name,) in tables])

for table in ("account", "loan", "district"):
    columns = conn.execute(f'PRAGMA table_info("{table}")').fetchall()
    print(f"\n{table}:", [(col[1], col[2]) for col in columns])

Tables: ['account', 'card', 'client', 'disp', 'district', 'loan', 'order', 'trans']

account: [('account_id', 'INTEGER'), ('district_id', 'INTEGER'), ('frequency', 'TEXT'), ('date', 'DATE')]

loan: [('loan_id', 'INTEGER'), ('account_id', 'INTEGER'), ('date', 'DATE'), ('amount', 'INTEGER'), ('duration', 'INTEGER'), ('payments', 'REAL'), ('status', 'TEXT')]

district: [('district_id', 'INTEGER'), ('A2', 'TEXT'), ('A3', 'TEXT'), ('A4', 'TEXT'), ('A5', 'TEXT'), ('A6', 'TEXT'), ('A7', 'TEXT'), ('A8', 'INTEGER'), ('A9', 'INTEGER'), ('A10', 'REAL'), ('A11', 'INTEGER'), ('A12', 'REAL'), ('A13', 'REAL'), ('A14', 'INTEGER'), ('A15', 'INTEGER'), ('A16', 'INTEGER')]


In [5]:
# How many accounts in the Prague region are eligible for loans?

reference_sql = """
SELECT COUNT(t1.account_id)
FROM account AS t1
INNER JOIN loan AS t2 ON t1.account_id = t2.account_id
INNER JOIN district AS t3 ON t1.district_id = t3.district_id
WHERE t3.a3 = 'Prague'
"""
print("Reference result:", conn.execute(reference_sql).fetchall())

Reference result: [(84,)]


In [4]:
# Inspect how Prague is actually stored
prague_values = conn.execute("""
    SELECT DISTINCT A3
    FROM district
    WHERE LOWER(A3) LIKE '%prague%'
""").fetchall()

print("Matching region values:", prague_values)

# Compare exact and case-insensitive filters
comparison = conn.execute("""
    SELECT
        SUM(CASE WHEN d.A3 = 'prague' THEN 1 ELSE 0 END) AS exact_lowercase,
        SUM(CASE WHEN LOWER(d.A3) = 'prague' THEN 1 ELSE 0 END) AS case_insensitive
    FROM account AS a
    JOIN loan AS l ON a.account_id = l.account_id
    JOIN district AS d ON a.district_id = d.district_id
""").fetchone()

print("Exact lowercase vs case-insensitive:", comparison)

Matching region values: [('Prague',)]
Exact lowercase vs case-insensitive: (0, 84)
